In [1]:
import pandas as pd

In [2]:
!pip install pandas

## Create a dataframe views with two columns: datetime and user by reading feed-views.log
* convert the datetime to the datetime64[ns] Dtype
* extract the year, month, day, hour, minute, and second from the values of that column to the new columns

In [2]:
views = pd.read_csv("../data/feed-views.log", sep="\t", names=["datetime", "user"])
views["datetime"] = views["datetime"].astype("datetime64[ns]")


views.count()

datetime    1076
user        1076
dtype: int64

In [28]:
views["year"] = views["datetime"].dt.year
views["month"] = views["datetime"].dt.month
views["day"] = views["datetime"].dt.day
views["hour"] = views["datetime"].dt.hour
views["minute"] = views["datetime"].dt.minute
views["second"] = views["datetime"].dt.second

views.head(5)

,datetime,user,year,month,day,hour,minute,second
0,2020-04-17 12:01:08.463179,artem,2020,4,17,12,1,8
1,2020-04-17 12:01:23.743946,artem,2020,4,17,12,1,23
2,2020-04-17 12:27:30.646665,artem,2020,4,17,12,27,30
3,2020-04-17 12:35:44.884757,artem,2020,4,17,12,35,44
4,2020-04-17 12:35:52.735016,artem,2020,4,17,12,35,52


## Create the new column daytime
* you need to assign the particular time of day value if an hour is within a particular interval, for example, afternoon if the hour is larger than 11 and less or equal to 17
* 0 – 3.59 night, 4 – 6.59 early morning, 7 – 10.59 morning, 11 – 16.59 afternoon, 17 – 19.59 early evening, 20 – 23.59 evening
* use the method cut to solve this subtask
* assign the column user as the index

In [29]:
bins=[0, 4, 7, 11, 17, 20, 24]
labels = ["night", "early morning", "morning", "afternoon", "early evening", "evening"]
views["daytime"] = pd.cut(views["hour"], bins=bins, labels=labels, right=False)

views.set_index("user", inplace=True)

views.head(10)

,datetime,year,month,day,hour,minute,second,daytime
user,,,,,,,,
artem,2020-04-17 12:01:08.463179,2020,4,17,12,1,8,afternoon
artem,2020-04-17 12:01:23.743946,2020,4,17,12,1,23,afternoon
artem,2020-04-17 12:27:30.646665,2020,4,17,12,27,30,afternoon
artem,2020-04-17 12:35:44.884757,2020,4,17,12,35,44,afternoon
artem,2020-04-17 12:35:52.735016,2020,4,17,12,35,52,afternoon
oksana,2020-04-17 12:36:21.401412,2020,4,17,12,36,21,afternoon
oksana,2020-04-17 12:36:22.023355,2020,4,17,12,36,22,afternoon
artem,2020-04-17 13:55:19.129243,2020,4,17,13,55,19,afternoon
artem,2020-04-17 15:00:33.138530,2020,4,17,15,0,33,afternoon


## Calculate the number of elements in your dataframe
* use the method count()
* calculate the number of elements in each time of day category using the method value_counts()
* sort values in your dataframe by hour, minute, and second in ascending order (simultaneously and not one by one)


In [30]:
views.count()

datetime    1076
year        1076
month       1076
day         1076
hour        1076
minute      1076
second      1076
daytime     1076
dtype: int64

In [31]:
views["daytime"].value_counts()

daytime
evening          509
afternoon        252
early evening    145
night            129
morning           36
early morning      5
Name: count, dtype: int64

In [33]:
views.sort_values(by=["hour", "minute", "second"], inplace=True)

views.head(5)

,datetime,year,month,day,hour,minute,second,daytime
user,,,,,,,,
valentina,2020-05-15 00:00:13.222265,2020,5,15,0,0,13,night
valentina,2020-05-15 00:01:05.153738,2020,5,15,0,1,5,night
pavel,2020-05-12 00:01:27.764025,2020,5,12,0,1,27,night
pavel,2020-05-12 00:01:38.444917,2020,5,12,0,1,38,night
pavel,2020-05-12 00:01:55.395042,2020,5,12,0,1,55,night


## Calculate the minimum and maximum for the hours and the mode for the daytime categories
* calculate the maximum of hour for the rows where the time of day is night
* calculate the minimum of hour for the rows where the time of day is morning
* In addition to this, find out who visited the page at those hours (make one example from that)
* calculate the mode for the hour and daytime


In [34]:
agg_func = {"hour": ["min", "max"]}
views.groupby("daytime").agg(agg_func)

/var/folders/jb/zkfyxcwx1_z7_qyw8llwwzxw0000gn/T/ipykernel_99629/2612717245.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  views.groupby("daytime").agg(agg_func)


hour    
               min max
daytime               
night            0   3
early morning    4   4
morning          8  10
afternoon       11  16
early evening   17  19
evening         20  23

In [35]:
max_hour_night = views[views["daytime"] == "night"]["hour"].max()
min_hour_morning = views[views["daytime"] == "morning"]["hour"].min()

print(max_hour_night, min_hour_morning)

3 8


In [38]:
views[views["hour"] == max_hour_night].index[0]


'konstantin'

In [39]:
views[views["hour"] == min_hour_morning].index[0]

'alexander'

In [40]:
views["hour"].mode()

0    22
Name: hour, dtype: int32

In [41]:
views["day"].mode()

0    11
Name: day, dtype: int32

## Show the 3 earliest hours in the morning and the corresponding usernames and the 3 latest hours and the usernames using nsmallest() and nlargest()

In [49]:
views[views["daytime"] == "morning"].nsmallest(3, ["hour"])


,datetime,year,month,day,hour,minute,second,daytime
user,,,,,,,,
alexander,2020-05-15 08:16:03.918402,2020,5,15,8,16,3,morning
alexander,2020-05-15 08:35:01.471463,2020,5,15,8,35,1,morning
alexander,2020-05-15 09:02:24.999438,2020,5,15,9,2,24,morning


In [52]:
views.nlargest(3, ["hour"])

,datetime,year,month,day,hour,minute,second,daytime
user,,,,,,,,
ekaterina,2020-05-14 23:02:11.327532,2020,5,14,23,2,11,evening
ekaterina,2020-05-14 23:02:14.494985,2020,5,14,23,2,14,evening
ekaterina,2020-05-14 23:02:15.588808,2020,5,14,23,2,15,evening


## Use the method describe() to get the basic statistics for the columns
* to find out what the most popular interval for visiting the page is, calculate the interquartile range for the hour by extracting values from the result of the describe() method and store it in the variable iqr


In [51]:
views.describe()

,datetime,year,month,day,hour,minute,second
count,1076,1076.0,1076.000000,1076.000000,1076.000000,1076.000000,1076.000000
mean,2020-05-10 09:00:41.211420672,2020.0,4.870818,13.552974,16.249071,29.629182,29.500929
min,2020-04-17 12:01:08.463179,2020.0,4.000000,1.000000,0.000000,0.000000,0.000000
25%,2020-05-10 01:13:49.857472,2020.0,5.000000,11.000000,13.000000,14.000000,14.000000
50%,2020-05-11 22:48:35.302552832,2020.0,5.000000,13.000000,19.000000,29.000000,30.000000
75%,2020-05-14 14:44:34.749530624,2020.0,5.000000,15.000000,22.000000,46.000000,45.000000
max,2020-05-22 10:36:14.662600,2020.0,5.000000,30.000000,23.000000,59.000000,59.000000
std,NaN,0.0,0.335557,4.906567,6.955490,17.689388,17.405506


In [54]:
iqr = views["hour"].describe().loc["75%"] - views["hour"].describe().loc["25%"]

iqr

np.float64(9.0)